from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()

llm = None

In [5]:
class JokeState(TypedDict):
    topic : str
    joke: str
    explanation: str

In [6]:
def generate_joke(state:JokeState) -> JokeState:
    prompt = f'Generate a joke on the topic {state["topic"]}.'
    response = llm.invoke(prompt).content
    return {'joke' : response}

In [7]:
def generate_explanation(state: JokeState):
    prompt = f'write an explanation for this joke {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation' : response}

In [ ]:
graph = StateGraph(JokeState)

graph.add_mode('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_job')
graph.add_edge('generate_job', 'generate_exxplanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer)

In [ ]:
config1 = {"configurable" : {"thread_id" : "1"}}
workflow.invoke({'topic' : 'pizza'}, config = config1)

In [ ]:
workflow.get_state(config1)

In [ ]:
list(workflow.get_state_history(config1))

In [ ]:
config2 = {"configurable" : {"thread_id" : "2"}}
workflow.invoke({'topic' : 'pasta'}, config = config2)

In [ ]:
workflow.get_state(config1)

In [ ]:
list(workflow.get_state_history(config1))

Fault Tolerance

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
class CrashState(TypedDict):
    input : str
    step1 : str
    step2 : str

In [ ]:
def step_1(state: CrashState) -> CrashState:
    print("step 1 executed")
    return {"step1" : "done", "input" : state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("step2... hanging ..... now manually from the notebook toolbar")
    time.sleep(1000)
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("step 3 executed")
    return {"done" : True}


In [ ]:
builder = StateGraph(CrashState)

builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compiler(checkpointer = checkpointer)

In [ ]:
try:
    print("running graph")
    graph.invoke({"input" : "start"}, config = {"configurable" : {"thread_id" : 'thread-1'}})

except keyboardInterrupt:
    print("kernel manually stopped by the user")
    

In [ ]:
print("re-running the demonstration")
final_state = graph.invoke(None, config = {"configurable": {"thread_id": 'thread-1'}})
print("final state", final_state)

In [ ]:
list(graph.get_state_history{"configurable" :{"thread_id" : 'thread-1'}}))